In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

## M2M Identity Middleware Test — Hi Genie Orchestrator

This notebook checks the residual risk flagged in Phase 3a of the R1 readiness work
(`hey_polly/001_release-1-readiness.md`): does the `requireVerifiedIdentity` middleware
(`server/api/middleware/identity.ts`) correctly see an `x-forwarded-user`-style header for
service-principal (M2M) callers, or does it reject that traffic where it previously
succeeded as the shared `'system'` identity?

**Flow (mirrors the SPN `client_credentials` pattern used elsewhere, e.g. the
dbxWearables `Authentication Test` notebook):**
1. Get the App's bootstrap SPN `client_id`/`client_secret` from the `hi_genie_credentials`
   secret scope.
2. Exchange them for a short-lived Databricks OAuth token via `POST /oidc/v1/token`
   (`grant_type=client_credentials`).
3. Call the deployed dev app's `GET /api/v1/plans/<bogus-uuid>` with
   `Authorization: Bearer <token>` — a side-effect-free read that still passes through
   `requireVerifiedIdentity`.
4. Interpret the result:
   - **401 `UNAUTHENTICATED`** → the middleware did NOT see an identity header for this
     SPN call. This is the regression risk — M2M traffic that used to succeed as
     `'system'` would now be rejected.
   - **404 `NOT_FOUND`** → the middleware saw an identity header, attached
     `req.verifiedIdentity`, and the request reached the route handler (which then
     correctly reports the bogus plan ID doesn't exist). No regression.
   - Anything else (500, connection error) → investigate separately; not a clean signal
     either way.

In [0]:
from databricks.sdk import WorkspaceClient

wc = WorkspaceClient()
WORKSPACE_HOST = wc.config.host.rstrip('/')

# ── App SPN credentials (dev target, schema-qualified per PROJECT_MEMORY.md) ──
SCOPE = 'hi_genie_credentials'
SPN_CLIENT_ID = dbutils.secrets.get(SCOPE, 'client_id_dev_matthew_giglia_hi_genie')
SPN_CLIENT_SECRET = dbutils.secrets.get(SCOPE, 'client_secret_dev_matthew_giglia_hi_genie')

# ── App URL (discovered from SDK, not hardcoded) ──
APP_NAME = 'mcp-hi-genie-orchestrator-dev'
APP_URL = None
for app in wc.apps.list():
    if app.name == APP_NAME:
        APP_URL = app.url.rstrip('/')
        break
assert APP_URL, f'App {APP_NAME} not found!'

print(f'Workspace:   {WORKSPACE_HOST}')
print(f'App URL:     {APP_URL}')
print(f'SPN Client:  {SPN_CLIENT_ID[:8]}...')

In [0]:
import requests

token_url = f'{WORKSPACE_HOST}/oidc/v1/token'
resp = requests.post(token_url, data={
    'grant_type': 'client_credentials',
    'client_id': SPN_CLIENT_ID,
    'client_secret': SPN_CLIENT_SECRET,
    'scope': 'all-apis',
}, timeout=15)

assert resp.status_code == 200, f'OAuth failed: {resp.status_code} {resp.text}'
oauth = resp.json()
SPN_TOKEN = oauth['access_token']

print('\u2713 SPN OAuth token acquired')
print(f'  Token type:  {oauth["token_type"]}')
print(f'  Expires in:  {oauth["expires_in"]}s')
print(f'  Token:       {SPN_TOKEN[:20]}...')

In [0]:
import uuid

bogus_plan_id = str(uuid.UUID(int=0))  # 00000000-0000-0000-0000-000000000000
test_url = f'{APP_URL}/api/v1/plans/{bogus_plan_id}'

resp = requests.get(
    test_url,
    headers={'Authorization': f'Bearer {SPN_TOKEN}'},
    timeout=30,
)

print(f'GET {test_url}')
print(f'Status: {resp.status_code}')
try:
    print(f'Body:   {resp.json()}')
except Exception:
    print(f'Body:   {resp.text[:500]}')

print()
if resp.status_code == 401:
    print('\u26a0\ufe0f  REGRESSION RISK CONFIRMED: requireVerifiedIdentity rejected this SPN-authenticated')
    print('    call. The Apps auth sidecar did not forward an x-forwarded-user-style header for')
    print('    this M2M token, so this traffic pattern would now fail where it previously')
    print('    succeeded as the shared \'system\' identity. Needs a fix before merging Phase 3a.')
elif resp.status_code == 404:
    print('\u2713 No regression: the middleware saw a verified identity for this SPN call and let')
    print('  the request reach the route handler (which then correctly 404s on the bogus plan id).')
else:
    print(f'\u2139\ufe0f  Unexpected status {resp.status_code} \u2014 investigate separately; not a clean')
    print('   pass/fail signal for the identity middleware question.')

In [0]:
import requests
import uuid as uuid_lib

# Validates migration 005 (dispatch.plans / dispatch.plan_snapshots RLS) end-to-end:
# create a real plan and confirm the write succeeds with correct created_by
# attribution, proving the SET LOCAL-equivalent set_config('app.user_identity', ...)
# + INSERT policy didn't break legitimate writes.
#
# Uses the SPN client_credentials token from Step 1 (already proven to reach the
# app's identity middleware — see the 404 result above) rather than
# wc.config.authenticate(), which returns a notebook-execution-scoped token that
# is not valid for a Databricks App's own HTTP ingress for a human/U2M identity
# (human app access normally goes through the browser SSO session, not a bearer
# token minted for platform REST API calls).
headers = {'Authorization': f'Bearer {SPN_TOKEN}'}
headers['Content-Type'] = 'application/json'

create_resp = requests.post(
    f'{APP_URL}/api/v1/plans',
    headers=headers,
    json={'project_id': str(uuid_lib.uuid4()), 'name': 'Phase 3b RLS validation plan'},
    timeout=30,
)
print(f'POST {APP_URL}/api/v1/plans')
print(f'Status: {create_resp.status_code}')
print(f'Body:   {create_resp.text[:500]}')

if create_resp.status_code == 201:
    plan = create_resp.json()['plan']
    print()
    print(f"\u2713 Plan created: plan_id={plan['plan_id']}, created_by={plan['created_by']}")
else:
    print()
    print('\u2717 Create failed \u2014 investigate before treating Phase 3b as validated.')

In [0]:
headers_debug = wc.config.authenticate()
print(type(headers_debug))
print({k: (v[:25] + '...' if isinstance(v, str) and len(v) > 25 else v) for k, v in headers_debug.items()})